# Oracle schema row-count comparison

## Goal

This notebook demonstrates OracleSchemaComp with safe fake data, then provides an editable wrapper for live comparisons. The live path calls the same `rowcount_compare.main()` function as the CLI. It is read-only, but whole-schema mode can be expensive because Oracle must run `COUNT(*)` against every discovered object.

## Setup

The setup cell finds the repository whether Jupyter starts in the project root or in `notebooks/`. Adding `src/` and `runbooks/` to `sys.path` makes the existing project modules importable; `sys.path` is Python's ordered list of module-search directories.

In [ ]:
from __future__ import annotations

import csv
import sys
import tempfile
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "src" / "rowcount_compare.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from the OracleSchemaComp repository or its notebooks directory")

for module_dir in (PROJECT_ROOT / "src", PROJECT_ROOT / "runbooks"):
    module_path = str(module_dir)
    if module_path not in sys.path:
        sys.path.insert(0, module_path)

import rowcount_compare
from scenario_support import run_compare

def preview_rows(rows: list[dict[str, str]], limit: int = 10) -> list[dict[str, str]]:
    """Return a bounded copy suitable for notebook display."""
    return rows[:limit]


def status_totals(rows: list[dict[str, str]]) -> dict[str, int]:
    """Count report rows by comparison status."""
    totals: dict[str, int] = {}
    for row in rows:
        status = row["status"]
        totals[status] = totals.get(status, 0) + 1
    return totals

print("Project root:", PROJECT_ROOT)

## Fake Oracle demonstration

This runs the real argument parsing, schema discovery, comparison, and report-writing flow against the project's fake Oracle driver. It needs no database and uses only invented object names and counts.

In [ ]:
mview_name = "DEPARTMENT_TOTALS_MATERIALIZED_VIEW"
mview_container = "DEPARTMENT_TOTALS_MV_0001"
fake_outcome = run_compare(
    counts_a={
        "HR_PROD.EMPLOYEES": 100,
        "HR_PROD.ACTIVE_EMPLOYEES": 90,
        f"HR_PROD.{mview_name}": 12,
        "HR_PROD.LEGACY_AUDIT": 4,
    },
    counts_b={
        "HR_UAT.EMPLOYEES": 100,
        "HR_UAT.ACTIVE_EMPLOYEES": 90,
        f"HR_UAT.{mview_name}": 12,
    },
    whole_schema=True,
    schema_a="HR_PROD",
    schema_b="HR_UAT",
    objects_a=["EMPLOYEES", "LEGACY_AUDIT", mview_container],
    objects_b=["EMPLOYEES", mview_container],
    views_a=["ACTIVE_EMPLOYEES"],
    views_b=["ACTIVE_EMPLOYEES"],
    mviews_a={mview_name: mview_container},
    mviews_b={mview_name: mview_container},
)

In [ ]:
print(fake_outcome.stdout)
print("Exit code:", fake_outcome.exit_code)
print("Status totals:", status_totals(fake_outcome.full_report))
print("Full report preview:")
pprint(preview_rows(fake_outcome.full_report))
print("Differences preview:")
pprint(preview_rows(fake_outcome.differences_report or []))

assert fake_outcome.exit_code == 1
assert status_totals(fake_outcome.full_report) == {"match": 3, "error": 1}

## Live Oracle parameters

Edit only this cell to configure a run. Credentials and DSNs are intentionally absent: the application reads them from exported environment variables or ignored `config/.env`. Never paste passwords into a notebook because notebook inputs and outputs are easy to share accidentally.

In [ ]:
RUN_LIVE_COMPARISON = False
COMPARISON_MODE = "whole-schema"  # whole-schema, single-table, or table-list
DB_A_SCHEMA = "HR"
DB_B_SCHEMA = "HR"
TABLE_NAME = "EMPLOYEES"
TABLES_FILE = PROJECT_ROOT / "config" / "tables.example.txt"
OUTPUT_DIR = PROJECT_ROOT / ".tmp" / "notebook-live-comparison"
LOG_LEVEL = "WARNING"

## Preview and validation

The builder validates the selected mode before any connection attempt and produces the exact argument list passed to the application.

In [ ]:
def build_live_arguments(
    mode: str,
    schema_a: str,
    schema_b: str,
    table_name: str,
    tables_file: Path,
    output_dir: Path,
    log_level: str,
) -> list[str]:
    """Validate notebook parameters and build rowcount_compare CLI arguments."""
    if mode == "whole-schema":
        if not schema_a.strip() or not schema_b.strip():
            raise ValueError("whole-schema mode requires DB_A_SCHEMA and DB_B_SCHEMA")
        arguments = ["--whole-schema"]
    elif mode == "single-table":
        if not table_name.strip():
            raise ValueError("single-table mode requires TABLE_NAME")
        arguments = ["--table", table_name.strip()]
    elif mode == "table-list":
        if not tables_file.is_file():
            raise ValueError(f"Table-list file not found: {tables_file}")
        arguments = ["--tables-file", str(tables_file)]
    else:
        raise ValueError("COMPARISON_MODE must be whole-schema, single-table, or table-list")

    if schema_a.strip():
        arguments.extend(["--db-a-schema", schema_a.strip()])
    if schema_b.strip():
        arguments.extend(["--db-b-schema", schema_b.strip()])
    arguments.extend(["--output-dir", str(output_dir), "--log-level", log_level])
    return arguments


sample_output = PROJECT_ROOT / ".tmp" / "notebook-argument-check"
assert build_live_arguments("whole-schema", "HR", "HR", "", TABLES_FILE, sample_output, "WARNING") == [
    "--whole-schema", "--db-a-schema", "HR", "--db-b-schema", "HR",
    "--output-dir", str(sample_output), "--log-level", "WARNING",
]
assert build_live_arguments("single-table", "HR", "HR", "EMPLOYEES", TABLES_FILE, sample_output, "INFO") == [
    "--table", "EMPLOYEES", "--db-a-schema", "HR", "--db-b-schema", "HR",
    "--output-dir", str(sample_output), "--log-level", "INFO",
]
assert build_live_arguments("table-list", "HR", "HR", "", TABLES_FILE, sample_output, "ERROR") == [
    "--tables-file", str(TABLES_FILE), "--db-a-schema", "HR", "--db-b-schema", "HR",
    "--output-dir", str(sample_output), "--log-level", "ERROR",
]
for invalid_call in (
    lambda: build_live_arguments("unknown", "HR", "HR", "", TABLES_FILE, sample_output, "WARNING"),
    lambda: build_live_arguments("whole-schema", "", "HR", "", TABLES_FILE, sample_output, "WARNING"),
):
    try:
        invalid_call()
    except ValueError:
        pass
    else:
        raise AssertionError("Expected invalid notebook parameters to raise ValueError")

print("Notebook parameter checks passed")

In [ ]:
live_arguments = build_live_arguments(
    COMPARISON_MODE, DB_A_SCHEMA, DB_B_SCHEMA, TABLE_NAME,
    TABLES_FILE, OUTPUT_DIR, LOG_LEVEL,
)
print("Arguments passed to rowcount_compare.main():")
print(live_arguments)

## Run live comparison

The saved notebook keeps live execution disabled. Review the argument preview, confirm the intended databases outside the notebook, then change `RUN_LIVE_COMPARISON` to `True` and run this cell.

In [ ]:
reports_before_live = {
    path: path.stat().st_mtime_ns
    for path in OUTPUT_DIR.glob("rowcount_*.csv")
} if OUTPUT_DIR.is_dir() else {}

if RUN_LIVE_COMPARISON:
    live_exit_code = rowcount_compare.main(live_arguments)
    print("Live comparison exit code:", live_exit_code)
else:
    live_exit_code = None
    print("Live comparison skipped. Set RUN_LIVE_COMPARISON = True to run it.")

## Inspect reports

Only the newest report for each pattern is loaded, and previews are capped at 10 rows to avoid embedding a large schema inventory in notebook output.

In [ ]:
def read_report(path: Path) -> list[dict[str, str]]:
    """Read one CSV report."""
    if not path.is_file():
        return []
    with path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def current_run_report_rows(
    exit_code: int | None,
    output_dir: Path,
    files_before: dict[Path, int],
) -> tuple[list[dict[str, str]], list[dict[str, str]]]:
    """Load only reports created or modified by the current successful run."""
    if exit_code not in (0, 1) or not output_dir.is_dir():
        return [], []
    changed = {
        path for path in output_dir.glob("rowcount_*.csv")
        if files_before.get(path) != path.stat().st_mtime_ns
    }
    full_paths = [path for path in changed if path.name.startswith("rowcount_report_")]
    if len(full_paths) != 1:
        raise RuntimeError(f"Expected one current full report, found {len(full_paths)}")
    full_path = full_paths[0]
    run_id = full_path.stem.removeprefix("rowcount_report_")
    differences_path = output_dir / f"rowcount_differences_{run_id}.csv"
    differences = read_report(differences_path) if differences_path in changed else []
    return read_report(full_path), differences


with tempfile.TemporaryDirectory(dir=PROJECT_ROOT / ".tmp") as report_test_dir:
    report_test_path = Path(report_test_dir)
    old_difference = report_test_path / "rowcount_differences_OLD.csv"
    old_difference.write_text("status\nmismatch\n", encoding="utf-8")
    before_test_run = {old_difference: old_difference.stat().st_mtime_ns}
    current_full = report_test_path / "rowcount_report_CURRENT.csv"
    current_full.write_text("status\nmatch\n", encoding="utf-8")
    test_full, test_differences = current_run_report_rows(0, report_test_path, before_test_run)
    assert test_full == [{"status": "match"}]
    assert test_differences == []
    assert current_run_report_rows(2, report_test_path, before_test_run) == ([], [])

full_report_rows, difference_rows = current_run_report_rows(
    live_exit_code, OUTPUT_DIR, reports_before_live
)

print("Live report status totals:", status_totals(full_report_rows))
print("Full report preview:")
pprint(preview_rows(full_report_rows))
print("Differences preview:")
pprint(preview_rows(difference_rows))

## Checks

- Exit `0`: every row count matched.
- Exit `1`: at least one mismatch or per-object error was reported; the run itself completed.
- Exit `2`: configuration, connection, or discovery failed.

For setup examples, see `config/.env.example` and `config/tables.example.txt`. For diagnosis, see the README troubleshooting section. Whole-schema discovery uses Oracle `ALL_*` views; prefer direct least-privilege grants and do not grant broad catalog roles merely to make discovery work.

In [ ]:
for forbidden_option in ("--password", "--dsn", "--username"):
    assert forbidden_option not in live_arguments
assert len(preview_rows(fake_outcome.full_report)) <= 10
assert len(preview_rows(full_report_rows)) <= 10
print("Safety and preview checks passed")

## Next steps

Start with the fake demonstration, then review the live parameter and argument-preview cells. For a live run, verify `config/.env` locally, set `RUN_LIVE_COMPARISON = True`, and execute from the live cell downward. Keep generated reports under an ignored or access-controlled directory because object names and error messages can reveal schema metadata.